# Making my toughts bilingual with Agentic AI and Harness Engineering

Hi there! It's been a while :)

Lets start with the most obvious statement that, if you are here you probably know already: **I am brazilian**. However, I've been working in multinational companies for almost 5~7 years and I got used to write in english. I code in english, [my master thesis](https://teses.usp.br/teses/disponiveis/45/45134/tde-25092025-141609/publico/MScThesisAESAndreBarbosaReviewed.pdf) was written in english; my grammar is usually not perfect but hey I try my best to improve my skills :)

Then, I also believe that english writing increase my technical reach. These days I have colleagues that arent brazilian and I'd like if they could read my think process. However, something that really annoyed me is that what about more junior people? When I entered university my english skills were poor and it would be awesome if I could have access to some posts in brazilian portuguese.

The challenge, though is what should I do to make my ideas more accessible? 

- Should I write in one language (the one that I wanted to express my idea first?) and then re-write it in another language? The effort is doubled, doesnt make sence
- Should I rely on a Translation API? Nah, I didnt want to pay anything for a part time hobby.

Wait, I'm a trained Data Scientist! I like languages and NLP, so I could build a model that translates my blog posts automatically so I could review them in the target answer after I write the first one originally. 

More than building, I could leverage on existing pre-trained models; with the help of AI Agents, how could I cheaply rely on them to build my experiments so I can choose a model and build a pipeline to help me achieve this goal? This is the story that I want to tell here :)

# Augmenting Human Capabilities, not replacing them

I remembered this claim from a previous manager that I had in my career. We were once work in a product to help customer support area of our company and I always raised the concern that we should build AI that are not replacing humans and if we are doing this, we should be responsible and acknowledgeable in our thinking/product development process. He pointed me [this article](https://hbr.org/2021/03/ai-should-augment-human-intelligence-not-replace-it), which framed a lot of my reasoning when I think about AI products.

Well, what is the relationship of this post? Id like a process that doesnt automatically translates something to a new one, but actually a process that I should be able to audit, but it is autonomous in some way that doesnt harm my development flow (again, writing this blog is a spare time process and I shouldnt waste much time on it). How could I achieve it? Github Actions!

## Architeture at a Glance

Briefly speaking, the pipeline will look like this*

![](images\scaling-translation-posts\architeture-automate-translation-v2.png)

*this has been build with the help of ChatGPT Sol with High Reasoning


The image is preety straight forward, but lets make sure we are in the same page: I am writing the post and by the time I do create a PR I want to setup an action that create the translated version of this same post in a given language. This translated version should be built by an CI step, so it need to host a model that is cheap/can run on CPU but also can create a temporary page so I can review the content easily and then everything is deleted once I merge the translated PR

The challenge? Find a reliable model

# The Search for a Good Model

I could blindly use a self hosted model from HuggingFace, but as a scientist I need to know how good the model is :) Then, what is the best approach to achieve this? evals!

## Constructing a reference dataset

I didnt want a iteris lipsis translation, but instead Id like a a translation that made sense. Fortunately, I tried to have a single use case translated in the past, i.e. [English Version](posts\2020-09-19-Distilling-BERT.ipynb) and [Portuguese Version](posts\.ipynb_checkpoints\2020-09-19-Distilling-BERT-pt-checkpoint.ipynb)... A post is a collection of sentences and thus I already have a dataset! In other words, I have a set of alinged sentence pairs between english and portuguese.

### The metrics

I wanted to rely on some skills that I learned from [AI Evals book](https://www.oreilly.com/library/view/evals-for-ai/9798341660717/) and, for this, I wanted to create some initial LLM-judges so we could rely on. But what to measure?

#### MQM

With the help of ChatGPT, I found this [a framework for Translation Quality Evaluation](https://themqm.org/error-types-2/values-and-scores/) that essentially uses AI Judges to get the quality of a translation. In a nutshell, it tries to answer this question: _What kinds of translation errors occurred, and how serious were they?_


The judge compares the source, candidate, and human reference. It marks errors by category and severity. Our categories include accuracy, omission, addition, fluency, terminology, locale, style, and formatting.

**self note: Id need to add a box saying this was my implementation of the framework** 

Our formula is:
$
[
P_s = 1N_{\text{minor}} + 5N_{\text{major}} + 10N_{\text{critical}}
]
$

Example: a segment with two minor terminology problems and one major omission receives:

$
[
2(1)+1(5)=7
]
$

Lower is better; zero means that the judge reported no errors.

#### Pairwise Preference

To calibrate the judges, we also need to correlate this metrics with a real perception by comparing translated content by models. Then we have this question: _If two translations are placed side by side, which one is better overall?_

For each source segment, the judge chooses candidate A, candidate B, or a tie. We run the comparison twice, reversing the candidate order.

A comparison is stable only if reversing the display order produces the same underlying result. For example:

- Model $X$ as A vs. Model $Y$ as B → B wins
- Model $Y$ as A vs. Model $X$ as B → A wins

Both judgments mean Model $Y$ won, so the comparison is stable.

The preference rate is:
$
[
R_m =
\frac{\text{stable comparisons won by model }m}
{\text{stable comparisons involving model }m}
]
$

Higher is better. Unstable comparisons are reported separately because they indicate order sensitivity or judge uncertainty.

Both model propmpts are here **TODO_ADD**; these are single shot and I wanted to use the best-but-cheap available model to use as Judges; for this, Ive chosen Kimi K3 under a budget of 20 dollars (~100 brazilian reais)

### Human judge Agreement

Question: Does the automated judge make decisions similar to our human reviewer (i.e. myself)?

Raw agreement is:

$
[
A = \frac{\text{human–judge matches}}{\text{reviewed stable items}}
]
$

We also calculate Cohen’s kappa:
$
[
\kappa = \frac{p_o-p_e}{1-p_e}
]
$

where ($p_o$) is observed agreement and ($p_e$) is agreement expected from the reviewers’ label frequencies.

Raw agreement is easy to understand. Kappa is useful because two evaluators might agree frequently merely because both usually select the same label.

## Calibrating Judges and having a baseline

Once the Prompt and Judgemental Process are done I have the E2E pipeline done. Then, duh, I need the model candidates that will be the translator models. For this experiment, I did some quick research and I got three cadidates:
- Marian Models (TODO: add HuggingFace weights)-- this are extremely cheap, lightweight and I remembering playing with them when I worked in [bergamot project](https://browser.mt/)
- NLLB models (TODO: add reference)*
- Tower+ Models (TODO: add reference)**


* I qucikly dropped NLLB model as at least with a quick and dirty setup the model was **not** reliable and constantly generated repeated sentences and fell into loops

** As my final plan is to use these through free GitHub actions, I need the inference to happen on **CPU** so even though it is slow, I needed the models to run on non-GPU environments. Moreover, considering the Github free environment this is also why I used the 2B parameters even though I do know there are bigger models available

# Building the Evaluation Dataset

I started with two different notebooks. They might have the same information, but due to nuances of each language they are not necessairly side-by-side aligned. I could have paraphrased something, changed the Markdown because it made more sense in portuguese and so on. Therefore, simply matching sentence position could have generated incorrect source-sentence pairs.

To map and align English-Portuguese pairs I used LaBSE embeddings (**L**anguage **a**gnostic **B**ERT **S**entence **E**mbeddigns). For an English sentence (e) and Portuguese sentence (p), we calculated approximately:

$

[
\text{similarity}(e,p)

\cos\bigl(\operatorname{LaBSE}(e),\operatorname{LaBSE}(p)\bigr)
]

$

A high cosine similarity suggests that the passages express similar meanings even though they contain no words in common.

For example:

> We use the CLS token representation.

and:

> Utilizamos a representação do token CLS.

have little surface overlap, but LaBSE should place them close together because their meanings correspond.

## Expanding Embeddings

Codex GPT 5.6 Sol suggested me this approach. I liked and found it really smart so I also asked it to explain why this is necessary. Here it is a paraphased answer:

Selecting the most similar Portuguese sentence independently for every English sentence could reuse sentences or scramble their order. Therefore, LaBSE supplied the **semantic similarity signal**. We still need to get the best **global** alingment and a good way to ahcieve this is through Dynamic Programming.


The alignment algorithm enforced:

- Monotonic order: later English content maps to later Portuguese content.
- No overlapping or reused passages.
- One-to-many and many-to-one alignments, up to three sentences per side.
- Gaps for content present in only one language.

Conceptually, each candidate alignment received:

$
[
\text{alignment score}

\text{LaBSE similarity}
-\text{merge penalty}
-\text{length mismatch penalty}
]
$

The algorithm then found the sequence of alignments with the greatest total score. Gaps received their own negative penalty.

This is implemented in [scripts/translation_eval.py:304](scripts/translation_eval.py:304)

### Making Judgements Trustworthy

In order to make sure that I could rely on judges in a trustworthy manner, I asked codex to create a UI following the guidance of [AI Engineers eval](add linkg) book. Then, I could perform a human baseline review knowing if I would need to calibrate the prompts/choose another model and how well the judges would reflect my decision. A sample of the UI can be checked below:

![](images\scaling-translation-posts\ui-review-translate.jpg)


As you can see, I have an initial text/sentence the portuguese counterpart and then I could overwride (if I find something should be fixed) and then `Accept` if both sentences are well aligned; `Localize` if I paraphrased one of them, for example; `Exclude` from my evaluation set or `Defer`. Once I finished this, I would have an **reliable** and **trustworthy** evaluation set that I could run my Judges on

# Are the Judges Trustworthy?

My Judges ran on two environments: MQM and Pairwise. The former would be helpful to sport potential issues that I could fix myself and the latter is actually to measure how the Judge itself is reliable/agree with me

I wanted an MVP, so I biased myself towards known disagreements, close calls and easy cases. In a nuthsell, I wanted to, first, stress test the judge behavior. This resulted in 18 cases.

Of the 18 selected items:

- I reviewed 17 and deferred 1.
- Among my 17 completed items, 3 had unstable automated decisions.
- That left 14 completed items with stable judge answers.

Therefore, the reported agreement was:

$
[
\frac{11\text{ agreements}}{14\text{ comparable items}}=78.6%
]
$

Which is better than my initial quality gate of 70%! yey! If Id like to be even more rigourous I could compute Cohen Kappa, which in this case was $\kappa=0.672$. Literature defines above 0.6 as a good alingment (miss citation)

# Benchmarking the results

Now that I have a Judge that I know that I can trust a bit, I know that a zero-shot Kimi K3 is confident so I feel confident to run MQM Judges. Again, I removed NLLB due to its stability. Then, I wanted two benchmarks: EN->PT translatiosn and PT->EN translations as I want to write in both languages

  | Direction | Model | Mean MQM penalty | Median MQM penalty | Pairwise preference |
  |---|---|---:|---:|---:|
  | EN to PT-BR | Marian OPUS-MT | 3.67 | 3.0 | 20.6% |
  | EN to PT-BR | Tower+ 2B | 1.92 | **1.0** | **79.4%** |
  | PT-BR to EN | Marian OPUS-MT | 2.56 | 1.0 | 21.0% |
  | PT-BR to EN | Tower+ 2B | 0.69 | **0.0** | **79.0%** |

For MQM, lower is better. For pairwise preference, higher is better.

Tower+ produced fewer and less severe errors in both directions:

- EN → PT-BR: mean penalty fell from 3.67 to 1.92—a 48% reduction.
- PT-BR → EN: mean penalty fell from 2.56 to 0.69—a 73% reduction.

The median penalty for Tower+ in Portuguese → English was zero. This means at least half of its translations received no MQM penalty from the judge.

## MQM Error Analysis

Neither model received a critical error. Tower+ nevertheless reduced both minor and major errors substantially, with its strongest result in Portuguese → English.

  | Direction | Model | Minor errors | Major errors | Critical errors |
  |---|---|---:|---:|---:|
  | EN to PT-BR | Marian OPUS-MT | 47 | 17 | 0 |
  | EN to PT-BR | Tower+ 2B | 29 | 8 | 0 |
  | PT-BR to EN | Marian OPUS-MT | 32 | 12 | 0 |
  | PT-BR to EN | Tower+ 2B | 10 | 3 | 0 |

### Pairwise stability


  When the judge compared both translations directly, Tower+ was preferred approximately four out of five times:

  - EN → PT-BR: 79.4% Tower+ versus 20.6% Marian.
  - PT-BR → EN: 79.0% Tower+ versus 21.0% Marian.

| Direction | Stable comparisons | Unstable comparisons |
|---|---:|---:|
| EN to PT-BR | 34 | 2 |
| PT-BR to EN | 31 | 5 |

 Only stable comparisons contributed to the preference rate.

# Where the bilingual blog goes next

Through the metrics is clear that from a perspective of quality only, , Tower+ 2B produced better translations than Marian OPUS-MT in both directions, with a  Direct pairwise evaluation preferred Tower+ approximately 79% of the time in both directions. The model is heavy but fit in CPU and if you are seeing this blog post in potuguese it means this approached was deployable. Some ideas that Id like to do next:

- Reviewer Agent: deploy a decoder agent lightweight that at the time of the PR can also spot potential improvements in translation writing

- Faster models with better quality: The first model might be a 2B parameter that fits into CPU; however, I know that Marian models are faster and much ligher. As long as I write more I will have more data samples and as I evaluate I will slowly have a better dataset. Then I could either finetune a Tower model to fit my needs or fine tune Marian/Tower models; The same is valid for quantization models.


In this blog post we analyzed from a perspective of having access to powerful coding agents through Codex how code became cheap for me to create an evaluation dataset and get some numbers and how we as Humans could step in to build a AI prduct/feature that makes sense. Ah! I think it is important to considerer the time/money spent in the process of this writing:
- Time: 2 days
- Mondey: USD 40 dollars (20 for codex and 20 for my Kimi Judge, which still have a good budget left)

Until next time!